# 03 Noise test of the trained circuit Q4 (inference only)
Retrains the 25 q4 runs of plan A (5 folds x 5 seeds, same seed, fold, initial weights and code as the main runs), checks that the test Dice matches the stored run, and then adds synthetic depolarising, damping and readout noise with 1,024, 4,096, 8,192 shots or exact expectation values. No weight is refitted under noise.
Run cells 1 and 2 in each session. Several sessions can run cell 2 at the same time (set a different `SESSION`). T4 GPU only. About 20 to 25 minutes per run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time
ROOT = '/content/drive/MyDrive/BUSI_QB'
CODE = os.path.join(ROOT, 'code'); PREP = os.path.join(ROOT, 'prepared'); RUNS = os.path.join(ROOT, 'runs')
NOISE = os.path.join(ROOT, 'runs_noise')
sys.path.insert(0, os.path.join(CODE, 'src'))
SESSION = 'A'                                     # 'B', 'C' in other sessions
import numpy as np, pandas as pd, tensorflow as tf
import qsim, qbus, qnoise, runner, noise_run
print(qbus.env_info())
assert 'T4' in str(qbus.env_info().get('gpu')), 'select a T4 GPU runtime'
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print('determinism not available:', e)
import subprocess
t = subprocess.run([sys.executable, os.path.join(CODE, 'src', 'test_qnoise.py')], capture_output=True, text=True)
print(t.stdout, t.stderr); assert 'qnoise tests OK' in t.stdout
dec = json.load(open(os.path.join(RUNS, 'plan_decision.json'))); K = dec['selected']['q4']
todo = [dict(plan='A', model='q4', fold=f, seed=s, frac=1.0, k=K) for s in runner.SEEDS for f in range(5)]
print(len(todo), 'runs, k =', K)

In [ ]:
# 2. RUNS ------------------------------------------------------------------------------------------------
for r in runner.ordered(todo, SESSION):
    rid = runner.run_id(r)
    if runner.claim(NOISE, rid, SESSION):
        print(time.strftime('%m-%d %H:%M'), 'start', rid)
        s = noise_run.one_noise_run(r, PREP, RUNS, NOISE)
        print('   retrain Dice', round(s['retrain_dice'], 4), 'stored', round(s['original_dice'], 4),
              'max image diff', round(s['max_abs_image_diff'], 4), 'min', round(s['minutes'], 1))

In [ ]:
# 3. PROGRESS AND COLLECTION (run when all 25 are done) ----------------------------------------------------
done = glob.glob(os.path.join(NOISE, '*', 'summary.json')); locks = glob.glob(os.path.join(NOISE, '*.lock'))
print(f'done {len(done)} / 25, running {len(locks)}')
if len(done) == 25:
    S = pd.DataFrame([json.load(open(p)) for p in done]).drop(columns='env')
    print(S[['run_id', 'retrain_dice', 'original_dice', 'max_abs_image_diff', 'exact_via_code_path']].round(4).to_string())
    T = []
    for p in done:
        d = pd.read_csv(os.path.join(os.path.dirname(p), 'noise.csv')); s = json.load(open(p))
        d.insert(0, 'seed', s['seed']); d.insert(1, 'fold', s['fold']); T.append(d)
    T = pd.concat(T)
    os.makedirs(os.path.join(ROOT, 'analysis'), exist_ok=True)
    T.to_csv(os.path.join(ROOT, 'analysis', 'noise_q4.csv.gz'), index=False)
    S.to_csv(os.path.join(ROOT, 'analysis', 'noise_q4_runs.csv'), index=False)
    print(T.groupby(['level', 'shots']).dice.mean().unstack().round(4))
    print('NOISE COLLECT OK')